# **Initialization**

In [32]:
"""Start"""

'Start'

In [33]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from docplex.mp.model import Model

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\3_Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\3_Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Data**

In [34]:
# ==========================================
# 1. DATA READING
# ==========================================
def read_tsp_cappart_format(file_path):
    """Parses TSP text files."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
        
    with open(file_path, 'r') as f:
        values = f.read().split()

    iterator = iter(values)
    try:
        n = int(next(iterator))
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator))
                row.append(int(val))
            c.append(row)
        return n, c
    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

In [35]:
# Load Data Global Variables (simplest way for registries to access them)
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20\0.txt"

try:   
    number_of_customers, distance_list = read_tsp_cappart_format(base_path)
    print(f'Number of customer is {number_of_customers}')
    print(f'Distance matrix is {distance_list}')

except FileNotFoundError:
    print(f"Error: The file at {base_path} was not found.")

Number of customer is 20
Distance matrix is [[0, 34, 48, 30, 50, 10, 28, 41, 32, 4, 53, 62, 22, 59, 38, 36, 36, 37, 2, 13], [34, 0, 33, 4, 83, 34, 42, 69, 56, 35, 63, 30, 44, 39, 49, 5, 53, 70, 35, 31], [48, 33, 0, 35, 94, 41, 70, 90, 49, 46, 40, 55, 67, 72, 33, 28, 80, 73, 50, 54], [30, 4, 35, 0, 79, 30, 38, 65, 53, 31, 62, 34, 40, 40, 48, 8, 50, 66, 31, 27], [50, 83, 94, 79, 0, 54, 54, 28, 53, 50, 81, 109, 45, 97, 70, 86, 50, 27, 48, 54], [10, 34, 41, 30, 54, 0, 38, 50, 24, 7, 44, 64, 32, 64, 28, 35, 46, 37, 12, 22], [28, 42, 70, 38, 54, 38, 0, 31, 60, 32, 82, 58, 10, 43, 66, 47, 11, 56, 27, 16], [41, 69, 90, 65, 28, 50, 31, 0, 61, 44, 88, 90, 26, 73, 74, 73, 24, 43, 40, 38], [32, 56, 49, 53, 53, 24, 60, 61, 0, 28, 28, 86, 51, 89, 18, 55, 66, 27, 33, 45], [4, 35, 46, 31, 50, 7, 32, 44, 28, 0, 50, 64, 25, 62, 34, 36, 40, 36, 5, 17], [53, 63, 40, 62, 81, 44, 82, 88, 28, 50, 0, 91, 75, 101, 16, 60, 90, 54, 55, 66], [62, 30, 55, 34, 109, 64, 58, 90, 86, 64, 91, 0, 65, 27, 79, 32, 68, 99,

# **Model and dual bound declaration**

In [36]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    n = number_of_customers
    c = distance_list 
    
    # 2. Initialize Model
    # Note: Ensure float_cost matches your data. Your snippet used False (Int), 
    # so we explicitly cast distances to Int in the reader.
    model = m_dp.Model(maximize=False, float_cost=False)

    customer = model.add_object_type(number=n)

    # 3. State Variables
    # U: Unvisited set (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    # i: Current location
    location = model.add_element_var(object_type=customer, target=0)

    # 4. Resource Tables
    travel_time = model.add_int_table(c)

    # 5. Transitions
    # Visit customer j
    for j in range(1, n):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)

    # Return to depot
    # Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
        effects=[
            (location, 0),
        ],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    # 8. Create Bundle (Model + Metadata)
    # This metadata dict allows your heuristics (like MST or assignment) 
    # to access the raw matrix data later.
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
        # Add other keys if your dual bounds need them
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [37]:
def create_persistent_lp_relaxation_dual_bounds(metadata):
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    # Create the model instance only ONCE
    mdl = Model(name='Persistent_TSP_Relaxation')
    
    # Optimization Parameters for Speed
    mdl.parameters.threads = 1
    mdl.parameters.lpmethod = 1  # Primal Simplex is efficient for re-optimization
    mdl.log_output = False       # Silence output

    # --- Create Variables ---
    # x[i, j]: Flow variables (Continuous 0-1)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = mdl.continuous_var(lb=0, ub=1, name=f'x_{i}_{j}')

    # u[i]: MTZ potential variables
    u = {i: mdl.continuous_var(lb=0, ub=n_nodes, name=f'u_{i}') for i in range(n_nodes)}

    # --- Create Constraints (Store references to update them later) ---
    # We create constraints for ALL nodes initially.
    # We will toggle their RHS (Right Hand Side) between 1 and 0 dynamically.
    
    cons_out = {} # Constraint: Sum(x_ij) = RHS
    cons_in = {}  # Constraint: Sum(x_ji) = RHS
    
    for i in range(n_nodes):
        # Outgoing flow
        # sum(x[i, j] for all j) == RHS
        expr_out = mdl.sum(x[(i, j)] for j in range(n_nodes) if i != j)
        cons_out[i] = mdl.add_constraint(expr_out == 1, ctname=f'deg_out_{i}')
        
        # Incoming flow
        # sum(x[j, i] for all j) == RHS
        expr_in = mdl.sum(x[(j, i)] for j in range(n_nodes) if i != j)
        cons_in[i] = mdl.add_constraint(expr_in == 1, ctname=f'deg_in_{i}')

    # MTZ Constraints (Static - they rely on x and u values)
    # u[i] - u[j] + N * x[i,j] <= N - 1
    # We don't need to remove these; if x[i,j] is forced to 0, the constraint becomes loose (valid).
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            mdl.add_constraint(
                u[i] - u[j] + n_nodes * x[(i, j)] <= n_nodes - 1
            )

    # --- Static Objective ---
    obj_expr = mdl.sum(dist_matrix[i][j] * x[(i, j)] 
                    for i in range(n_nodes)
                    for j in range(n_nodes) if i != j)
    mdl.minimize(obj_expr)

    # ==========================================
    # 2. DYNAMIC HEURISTIC (Runs many times)
    # ==========================================
    def h_lp_relaxation(state):
        # A. Identify Active Nodes
        # The 'Active Subgraph' consists of: Current Node + Unvisited Nodes + Depot
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        # Quick exit for solved state
        if not unvisited and current_node == 0: 
            return 0.0

        # Construct a fast lookup set for active nodes
        # 0 (Depot) is always part of the formulation in this relaxation
        active_set = set(unvisited)
        active_set.add(current_node)
        active_set.add(0) 

        # B. Update Model (The Optimization)
        # Instead of rebuilding, we just switch the "power" on/off for nodes
        
        for i in range(n_nodes):
            if i in active_set:
                # ACTIVE NODE: Must have degree 1 (Flow = 1)
                cons_out[i].rhs = 1
                cons_in[i].rhs = 1
                # Ensure u-variable is active (allowed to be > 0)
                u[i].ub = n_nodes
            else:
                # INACTIVE NODE: Must have degree 0 (Flow = 0)
                # Setting RHS to 0 forces all connected x_ij variables to 0
                # because x_ij >= 0. This effectively removes the node.
                cons_out[i].rhs = 0
                cons_in[i].rhs = 0
                # Fix u-variable to 0 to help solver
                u[i].ub = 0

        # C. Solve Re-optimized Model
        # cplex/docplex is smart enough to use the previous basis for speed
        sol = mdl.solve()
        
        if sol:
            return float(sol.objective_value)
        return 0.0

    return h_lp_relaxation

def dual_bound_expression_function(didp_bundle):
    " Returns a dictionary of heuristic functions (bounds) bound to the model data."
    
    model, metadata = didp_bundle
    
    # Extract metadata
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) # Numpy version for calculations
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    num_nodes = metadata['num_nodes'] # Assumed available from creation function

    # --- Pre-computation for h_global_min_flow (Bound 2) ---
    # We calculate the global min outgoing and incoming edges for every node once.
    # This matches the 'min_to' and 'min_from' tables in the DIDP snippet.
    
    # masked_cost: diagonal is infinity to ignore self-loops
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    
    # min_outgoing[i] = min cost to leave node i
    min_outgoing_arr = np.min(masked_cost, axis=1)
    
    # min_incoming[j] = min cost to enter node j
    min_incoming_arr = np.min(masked_cost, axis=0)

    # ==========================================
    # 1. Degree Average Bound (Local Subgraph) [NEW]
    # ==========================================
    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        # If no unvisited nodes and we are at depot (0), cost is 0
        if not U and curr == 0:
            return 0.0

        # Define active nodes for the path: Current -> [Unvisited] -> Depot (0)
        # We need to construct the submatrix for these specific nodes
        active_nodes = [curr] + sorted(list(U))
        if 0 not in active_nodes:
            active_nodes.append(0)
            
        # Extract submatrix
        sub_mat = cost_matrix[np.ix_(active_nodes, active_nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)

        # Calculate mins within this specific subgraph
        # axis=0 is min down columns (Incoming), axis=1 is min across rows (Outgoing)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        
        # Logic for Path Constraints:
        # 1. Current Node (index 0 in active_nodes): Needs Outgoing, but NO Incoming
        # 2. Depot Node (index -1 in active_nodes): Needs Incoming, but NO Outgoing
        # 3. Intermediate (Unvisited): Need BOTH
        
        # Sum of valid Incoming edges (Everyone except Current)
        # Note: We must map the exclusion correctly. 
        # Since active_nodes[0] is 'curr', we exclude mins_in[0]
        sum_in = np.sum(mins_in[1:])
        
        # Sum of valid Outgoing edges (Everyone except Depot)
        # Since active_nodes[-1] is '0', we exclude mins_out[-1]
        sum_out = np.sum(mins_out[:-1])
        
        # Return average
        return float(0.5 * (sum_in + sum_out))

    # ==========================================
    # 2. Global Min Flow Bound (Max of Min-In/Out) [NEW]
    # ==========================================
    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        if not U and curr == 0:
            return 0.0

        # Bound A: Sum of minimum OUTGOING edges
        # We must leave 'curr' and every node in 'U'
        val_out = sum(min_outgoing_arr[u] for u in U)
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            
        # Bound B: Sum of minimum INCOMING edges
        # We must enter '0' and every node in 'U'
        val_in = sum(min_incoming_arr[u] for u in U)
        if curr != 0: # If we aren't already at 0, we must eventually enter 0
            val_in += min_incoming_arr[0]
            
        # Return the tighter (maximum) of the two constraints
        return float(max(val_out, val_in))

    # ==========================================
    # 3. LP Relaxation Bound (On-the-fly)
    # ==========================================
    h_lp_relaxation = create_persistent_lp_relaxation_dual_bounds(metadata = metadata)

    # ==========================================
    # 4. MST Bound
    # ==========================================
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    # ==========================================
    # 5. 1-Tree Bound
    # ==========================================
    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        subset_nodes = sorted(list(U))
        
        depot_edges = sorted(cost_matrix[0, subset_nodes])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset_nodes) > 1:
            sub_mat = cost_matrix[np.ix_(subset_nodes, subset_nodes)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    # ==========================================
    # 6. Assignment Bound
    # ==========================================
    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        # Fill diagonal with Infinity to forbid self-loops (i -> i)
        np.fill_diagonal(assign_mat, np.inf)
        # This finds the cheapest set of edges such that every row/col is used once
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    # ==========================================
    # 7. Eigenvalue Bound
    # ==========================================
    def h_eigen(state):
        U = state[unvisited_var]
        nodes = [0] + sorted(list(U))
        N = len(nodes)
        if N < 2: return 0.0

        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])

        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals):
                        phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
        return float(phi)
    
    # Return valid registry
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

In [ ]:
""" Sole LP relaxation
def create_persistent_lp_relaxation_dual_bounds(metadata):
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    # Create the model instance only ONCE
    mdl = Model(name='Persistent_TSP_Relaxation')
    
    # Optimization Parameters for Speed
    mdl.parameters.threads = 1
    mdl.parameters.lpmethod = 1  # Primal Simplex is efficient for re-optimization
    mdl.log_output = False       # Silence output

    # --- Create Variables ---
    # x[i, j]: Flow variables (Continuous 0-1)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = mdl.continuous_var(lb=0, ub=1, name=f'x_{i}_{j}')

    # u[i]: MTZ potential variables
    u = {i: mdl.continuous_var(lb=0, ub=n_nodes, name=f'u_{i}') for i in range(n_nodes)}

    # --- Create Constraints (Store references to update them later) ---
    # We create constraints for ALL nodes initially.
    # We will toggle their RHS (Right Hand Side) between 1 and 0 dynamically.
    
    cons_out = {} # Constraint: Sum(x_ij) = RHS
    cons_in = {}  # Constraint: Sum(x_ji) = RHS
    
    for i in range(n_nodes):
        # Outgoing flow
        # sum(x[i, j] for all j) == RHS
        expr_out = mdl.sum(x[(i, j)] for j in range(n_nodes) if i != j)
        cons_out[i] = mdl.add_constraint(expr_out == 1, ctname=f'deg_out_{i}')
        
        # Incoming flow
        # sum(x[j, i] for all j) == RHS
        expr_in = mdl.sum(x[(j, i)] for j in range(n_nodes) if i != j)
        cons_in[i] = mdl.add_constraint(expr_in == 1, ctname=f'deg_in_{i}')

    # MTZ Constraints (Static - they rely on x and u values)
    # u[i] - u[j] + N * x[i,j] <= N - 1
    # We don't need to remove these; if x[i,j] is forced to 0, the constraint becomes loose (valid).
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            mdl.add_constraint(
                u[i] - u[j] + n_nodes * x[(i, j)] <= n_nodes - 1
            )

    # --- Static Objective ---
    obj_expr = mdl.sum(dist_matrix[i][j] * x[(i, j)] 
                    for i in range(n_nodes)
                    for j in range(n_nodes) if i != j)
    mdl.minimize(obj_expr)

    # ==========================================
    # 2. DYNAMIC HEURISTIC (Runs many times)
    # ==========================================
    def h_lp_relaxation(state):
        # A. Identify Active Nodes
        # The 'Active Subgraph' consists of: Current Node + Unvisited Nodes + Depot
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        # Quick exit for solved state
        if not unvisited and current_node == 0: 
            return 0.0

        # Construct a fast lookup set for active nodes
        # 0 (Depot) is always part of the formulation in this relaxation
        active_set = set(unvisited)
        active_set.add(current_node)
        active_set.add(0) 

        # B. Update Model (The Optimization)
        # Instead of rebuilding, we just switch the "power" on/off for nodes
        
        for i in range(n_nodes):
            if i in active_set:
                # ACTIVE NODE: Must have degree 1 (Flow = 1)
                cons_out[i].rhs = 1
                cons_in[i].rhs = 1
                # Ensure u-variable is active (allowed to be > 0)
                u[i].ub = n_nodes
            else:
                # INACTIVE NODE: Must have degree 0 (Flow = 0)
                # Setting RHS to 0 forces all connected x_ij variables to 0
                # because x_ij >= 0. This effectively removes the node.
                cons_out[i].rhs = 0
                cons_in[i].rhs = 0
                # Fix u-variable to 0 to help solver
                u[i].ub = 0

        # C. Solve Re-optimized Model
        # cplex/docplex is smart enough to use the previous basis for speed
        sol = mdl.solve()
        
        if sol:
            return float(sol.objective_value)
        return 0.0

    return h_lp_relaxation

def dual_bound_expression_function(didp_bundle):
    " Returns a dictionary of heuristic functions (bounds) bound to the model data."
    
    model, metadata = didp_bundle
    
    # Extract metadata
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) # Numpy version for calculations
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    num_nodes = metadata['num_nodes'] # Assumed available from creation function

    # ==========================================
    # 3. LP Relaxation Bound (On-the-fly)
    # ==========================================
    h_lp_relaxation = create_persistent_lp_relaxation_dual_bounds(metadata = metadata)
    
    # Return valid registry
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())"""

# **EA Execution**

In [38]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 10        # Size of the population in each generation
GENERATIONS = 5           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 5 #seconds

In [39]:
params = EAHyperparameters(
    # --- 1. Population ---
    population_size=POPULATION_SIZE,          
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE,           

    # --- 2. Ranges & Constraints ---
    lb_range_of_constant=LB_range_of_constant,
    ub_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,     
    max_chromosome_length=max_chromosome_length,   

    # --- 3. Operator Specifics ---
    tournament_size=random.randint(2, 10),                             
    tournament_probability=tournament_probability,                    
    mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
    subtree_crossover_probability=subtree_crossover_probability,             
    uniform_crossover_probability=uniform_crossover_probability,             

    # --- 4. Problem Specific ---
    reference_point=OPTIMAL_COST_REFERENCE,         
    solver_time_limit=SOLVER_TIME_LIMIT,
    
    # Optional: You can override available operations if needed
    available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
)

print("--- Starting TSP Test Run ---")

# 2. Execution
best_individual = evolution_algorithm_execution(
    didp_model_registry=creation_of_didp_model_function,
    dual_bound_expression_function=dual_bound_expression_function,
    params=params
)
display(best_individual)

--- Starting TSP Test Run ---
--- Initialization: Generating Population of size 10 - 5 generations ---
Generating Initial Population at time: Sat Dec 13 10:00:00 2025
Initial Population Generated at time: Sat Dec 13 10:00:52 2025
Initial Best Fitness: 0.0175
Gen 1: Best Fitness = 0.0125 | Global Best = 0.0125
Gen 2: Best Fitness = 0.0125 | Global Best = 0.0125
Gen 3: Best Fitness = 0.0125 | Global Best = 0.0125
Gen 4: Best Fitness = 0.0125 | Global Best = 0.0125
Gen 5: Best Fitness = 0.0025 | Global Best = 0.0025

       PERFORMANCE PROFILING REPORT       
Total Runtime:    320.4770 seconds
--- Evolution Completed ---



{'chromosome': ['h_global_min_flow',
  'h_degree_average',
  'MULTIPLY',
  1.46,
  'h_degree_average',
  'MULTIPLY',
  'PDIV',
  'h_global_min_flow',
  2.94,
  'ADD',
  'MIN'],
 'fitness': 0.0025}

# **Test time log on 1-hour solver limit**

In [40]:
# 1. Create the Missing Registry
# We must instantiate the model once to get the heuristic functions dictionary
temp_bundle = creation_of_didp_model_function()
dual_bound_functions_registry = dual_bound_expression_function(temp_bundle)

# 2. Setup Chromosome
combined_dual_bound_chromosome = ['h_global_min_flow',
  'h_degree_average',
  'MULTIPLY',
  1.46,
  'h_degree_average',
  'MULTIPLY',
  'PDIV',
  'h_global_min_flow',
  2.94,
  'ADD',
  'MIN']
temp_dict = {'chromosome': combined_dual_bound_chromosome, 'fitness': 0}

print("The combined dual bounds will be depicted in the following code")
combined_dual_bound_function = compile_chromosome_to_useable_function(
    temp_dict, 
    dual_bound_functions_dict=dual_bound_functions_registry,
    print_code=True
)

print("\nSuccessfully created solver. Starting search...\n")

# 3. Run Solver
result = combining_modified_didppy_solver_with_chromosome(
    combined_dual_bound_chromosome, 
    creation_of_didp_model_function, 
    dual_bound_expression_function, 
    solver_time_limit=10,
    output_other_result=True,
    print_timing_stats=True
)

# If in Jupyter, use display(result), otherwise print(result)
try:
    display(result)
except NameError:
    print(result)

The combined dual bounds will be depicted in the following code
Generated Code:
def dual_bound_combination(state):
    return min(((h_global_min_flow(state) * h_degree_average(state)) / (1.46 * h_degree_average(state)) if abs((1.46 * h_degree_average(state))) > 1e-6 else 1.0), (h_global_min_flow(state) + 2.94))


Successfully created solver. Starting search...


                    📋 SOLVER RUN REPORT                     
🎯 SOLUTION STATUS:
   • Cost:              401
   • Status:            ⚠️  Suboptimal / Timeout
   • Nodes Generated:   10,066
   • Nodes Expanded:    3,108
   • Branching Factor:  ~3.24
------------------------------------------------------------
⏱️  TIME DISTRIBUTION (Total: 10.0296s):
   [Pure CABS time:   8.8%] 🆚 [Bridge time:  91.2%]

   1. 🟢 Pure CABS (Search):    0.8875 s
      └─ Average time per expanded state:   0.2855 ms
   2. 🔴 Total Bridge Time:     9.1421 s
      ├─ 🐍 Dual Bound Calculation Time: :     8.8767 s  ( 97.1% of bridge time)
      └─ 🌉 Switchi

None